In [14]:
!pip install -q torch datasets tqdm

In [15]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
import random

In [16]:
#Tiny Medical Dataset
text_data = """
fever is a common symptom of infection
headache can be caused by stress or illness
diabetes affects blood sugar levels in the body
hypertension means high blood pressure
cough is a reflex to clear the airways
medical diagnosis requires clinical examination
patients should drink water during fever
insulin is used to control diabetes
heart disease affects cardiovascular system
medicine should be taken as prescribed by doctor
"""

In [17]:
#Word tokenizer
words = text_data.lower().split()

vocab = sorted(set(words))

stoi = {w:i for i,w in enumerate(vocab)}
itos = {i:w for w,i in stoi.items()}

vocab_size = len(vocab)

def encode(text):
    return torch.tensor([stoi[w] for w in text.lower().split()], dtype=torch.long)

def decode(tokens):
    return " ".join([itos[i] for i in tokens])

In [18]:
#Dataset preparation
data = encode(text_data)

block_size = 32

def get_batch(batch_size=16):
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x, y

In [28]:
#GPT-style SLM
class MiniGPT(nn.Module):
    def __init__(self, vocab_size, n_embd=128, n_head=4, n_layer=2):
        super().__init__()

        self.token_emb = nn.Embedding(vocab_size, n_embd)
        self.pos_emb = nn.Embedding(512, n_embd)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=n_embd,
            nhead=n_head,
            batch_first=True
        )

        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=n_layer)
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, x):
        B, T = x.shape
        pos = torch.arange(T).unsqueeze(0).to(x.device)

        x = self.token_emb(x) + self.pos_emb(pos)

        x = self.transformer(x)

        logits = self.lm_head(x)
        return logits

In [20]:
#Initialize model
device = "cuda" if torch.cuda.is_available() else "cpu"

model = MiniGPT(vocab_size).to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)

In [21]:
#Training loop
for step in range(500):

    xb, yb = get_batch()
    xb, yb = xb.to(device), yb.to(device)

    logits = model(xb)

    loss = F.cross_entropy(logits.view(-1, vocab_size), yb.view(-1))

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if step % 50 == 0:
        print(f"Step {step}, Loss: {loss.item():.4f}")

Step 0, Loss: 4.1859
Step 50, Loss: 0.0468
Step 100, Loss: 0.0128
Step 150, Loss: 0.0068
Step 200, Loss: 0.0043
Step 250, Loss: 0.0034
Step 300, Loss: 0.0036
Step 350, Loss: 0.0021
Step 400, Loss: 0.0017
Step 450, Loss: 0.0021


In [22]:
#Text generation
def generate(model, start_text, max_new_tokens=100):

    model.eval()

    context = encode(start_text).unsqueeze(0).to(device)

    for _ in range(max_new_tokens):
        logits = model(context)

        probs = F.softmax(logits[:, -1, :], dim=-1)

        next_token = torch.multinomial(probs, 1)

        context = torch.cat([context, next_token], dim=1)

    return decode(context[0].tolist())

In [26]:
#Test your medical SLM
print(generate(model, "headache", 8))

headache can be caused by stress or illness diabetes


In [27]:
#Test your medical SLM
print(generate(model, "doctor", 8))

doctor the body hypertension means high blood sugar levels
